In [ ]:
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
import re

def extract_rdrp_domain(domain_file, nucleotide_file, output_file):
    domains_data = {}
    with open(domain_file, "r") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.strip().split("\t")
            protein_id = parts[0]
            domain_string = parts[2]
            domains_data[protein_id] = domain_string
    
    nucleotide_seqs = SeqIO.to_dict(SeqIO.parse(nucleotide_file, "fasta"))
    extracted = []
    
    for prot_id in domains_data:
        if prot_id in nucleotide_seqs:
            m3 = re.search(r"RdRP_3\|(\d+)\.\.(\d+)", domains_data[prot_id])
            m1 = re.search(r"RdRP_1\|(\d+)\.\.(\d+)", domains_data[prot_id])
            
            if m3:
                start_aa = int(m3.group(1))
                end_aa = int(m3.group(2))
                seq_record = nucleotide_seqs[prot_id].seq
                start_nt = (start_aa - 1) * 3
                end_nt = end_aa * 3
                dom_seq = seq_record[start_nt:end_nt]
                extracted.append(
                    SeqRecord(dom_seq, id=prot_id, description=f"RdRP_3 {start_aa}-{end_aa}"))
            
            if m1:
                start_aa = int(m1.group(1))
                end_aa = int(m1.group(2))
                seq_record = nucleotide_seqs[prot_id].seq
                start_nt = (start_aa - 1) * 3
                end_nt = end_aa * 3
                dom_seq = seq_record[start_nt:end_nt]
                extracted.append(
                    SeqRecord(dom_seq, id=f"{prot_id}_RdRP1", description=f"RdRP_1 {start_aa}-{end_aa}"))
    
    SeqIO.write(extracted, output_file, "fasta")
    print(f"Extracted {len(extracted)} domains")

extract_rdrp_domain(
    domain_file=r"C:\Users\2slon\OneDrive\Рабочий стол\5sem\pestiviruses\final\small\smal_search_pfam.domains",
    nucleotide_file=r"C:\Users\2slon\OneDrive\Рабочий стол\5sem\pestiviruses\final\small\nt_small_seqs.fasta",
    output_file=r"C:\Users\2slon\OneDrive\Рабочий стол\5sem\pestiviruses\final\small\NS5_cut.fasta"
)